# AnemiaLens Web di Google Colab

Notebook ini menjalankan **website AnemiaLens** (tampilan cek mata + model AI) di Colab dan memberi
**link HTTPS publik** yang bisa dibuka dari HP mana pun, cocok untuk demo ke juri.

**Cara pakai:** Runtime → *Run all*. GPU tidak wajib (CPU cukup, ±0,5 detik per foto).

Model diambil dari salah satu sumber ini (otomatis, urut):
1. `/content/work/model` bila notebook training masih aktif di sesi yang sama;
2. file `AnemiaLens_model*.zip` di `/content`;
3. jika tidak ada, Colab akan meminta Anda meng-upload `AnemiaLens_model_v3.zip` (ada di folder JOINTS 2026/AnemiaLens; berisi segmentasi v2).

In [ ]:
# Langkah 1 — pasang paket
!pip install -q timm==1.0.* segmentation-models-pytorch==0.5.* fastapi uvicorn python-multipart pillow-heif
import sklearn, numpy
print('scikit-learn', sklearn.__version__, '| numpy', numpy.__version__)

In [ ]:
# Langkah 2 — tulis kode aplikasi (paket anemialens + website)
import base64, io, zipfile, os
APP = '/content/AnemiaLens_web'
os.makedirs(APP, exist_ok=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode('...'))).extractall(APP)
print('Kode aplikasi siap di', APP)

In [ ]:
# Langkah 3 — siapkan model
import glob, os, shutil, tempfile, zipfile
MD = f'{APP}/hasil_model_v2/model'
os.makedirs(MD, exist_ok=True)
if os.path.exists('/content/work/model/thresholds.json'):
    shutil.copytree('/content/work/model', MD, dirs_exist_ok=True)
    print('Model diambil dari sesi training: /content/work/model')
else:
    zips = sorted(glob.glob('/content/AnemiaLens_model*.zip'))
    if not zips:
        from google.colab import files
        print('Upload AnemiaLens_model_v3.zip (tunggu sampai 100%)...')
        up = files.upload()
        zips = [f'/content/{k}' for k in up if k.endswith('.zip')]
    tmp = tempfile.mkdtemp()
    zipfile.ZipFile(zips[0]).extractall(tmp)
    src = next(os.path.dirname(p) for p in glob.glob(f'{tmp}/**/thresholds.json', recursive=True))
    shutil.copytree(src, MD, dirs_exist_ok=True)
    print('Model diambil dari', zips[0])
print(sorted(os.listdir(MD)))

In [ ]:
# Langkah 4 — jalankan server + link publik HTTPS
import re, subprocess, time, urllib.request, json, os
subprocess.run('pkill -f "web/server.py"; pkill -f cloudflared', shell=True)
srv = subprocess.Popen(['python', 'web/server.py', '--no-browser', '--port', '8000'], cwd=APP,
                       stdout=open('/content/server.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        info = json.load(urllib.request.urlopen('http://localhost:8000/api/info'))
        if info.get('ready') or info.get('error'):
            break
    except Exception:
        pass
    time.sleep(1)
print('Status model:', 'SIAP' if info.get('ready') else info.get('error'))

if not os.path.exists('/content/cloudflared'):
    subprocess.run('wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared', shell=True)
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
                       stdout=open('/content/tunnel.log', 'w'), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m:
        url = m.group(0); break
    time.sleep(1)
if url:
    print('\n' + '=' * 62)
    print('  BUKA DI HP / LAPTOP:  ' + url)
    print('=' * 62)
    print('Link aktif selama sel ini dan sesi Colab berjalan. Tunggu ±10 detik bila belum bisa dibuka.')
else:
    print('Link publik gagal dibuat, memakai jendela Colab (hanya untuk laptop ini):')
    from google.colab import output
    output.serve_kernel_port_as_window(8000)

In [ ]:
# (Opsional) lihat log server bila ada error
print(open('/content/server.log').read()[-3000:])

In [ ]:
# Matikan server & link publik
import subprocess
subprocess.run('pkill -f "web/server.py"; pkill -f cloudflared', shell=True)
print('Server dimatikan.')